# 01 — QoL_Stress dataset audit

Purpose: inspect the supplied CSV collections, participant linkage, labels, missingness and file integrity before selecting any modelling target. This notebook does not train a model and does not modify raw data.

Run from the `ml-workstream-2` project root. Dataset directory name is intentionally kept unchanged.

In [5]:
# CELL 1 — imports, project discovery, and paths
# Run this cell first. All libraries used later are imported here.

from pathlib import Path
import hashlib
import json
import re
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATASET_NAME = (
    "qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0"
)

# Leave as None for automatic discovery. If Jupyter is launched from an unrelated
# directory and discovery fails, set this to Path(r"D:\\...\\ml-workstream-2").
PROJECT_ROOT_OVERRIDE = None


def find_project_root(start: Path) -> Path:
    """Find the project root by locating its data/raw and notebooks folders."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the ml-workstream-2 project root.\n"
        f"Jupyter current working directory: {start}\n"
        "Expected a parent folder containing BOTH 'data/raw' and 'notebooks'.\n"
        "Set PROJECT_ROOT_OVERRIDE in this cell to the full path of ml-workstream-2, "
        "then rerun this cell."
    )


if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if not PROJECT_ROOT.is_dir():
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {PROJECT_ROOT}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / DATASET_NAME
DATA_DIR = DATASET_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

# Create output folders at the project root. This does not touch raw data.
for folder in (REPORTS_DIR, FIGURES_DIR, TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Dataset root :", DATASET_ROOT)
print("Data folder  :", DATA_DIR)
print("Reports      :", REPORTS_DIR)
print("Tables       :", TABLES_DIR)

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        "Dataset folder not found. Expected this exact folder:\n"
        f"{DATASET_ROOT}\n\n"
        "Check that the dataset was extracted under ml-workstream-2/data/raw/ "
        "and that the folder name has not been changed."
    )
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "The dataset root exists, but its 'data' subfolder is missing:\n"
        f"{DATA_DIR}\n"
        "The extracted dataset should contain a data/ directory."
    )

print("\nPath checks passed.")

Project root : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2
Dataset root : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\raw\qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0
Data folder  : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\raw\qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0\data
Reports      : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports
Tables       : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\tables

Path checks passed.


In [8]:
# CELL 2 — file inventory
# The eight expected collections are recognised both as *.csv files and as
# extensionless files (Windows may hide extensions in File Explorer).

EXPECTED_COLLECTION_STEMS = [
    "ecg",
    "eda",
    "fitbit_heart_rate_per_minute",
    "fitbit_heart_rate_per_second",
    "fitbit_heart_rate_summary",
    "participants",
    "pss_14",
    "stai",
]

all_data_files = sorted(p for p in DATA_DIR.iterdir() if p.is_file())

def normalised_stem(path: Path) -> str:
    return path.stem.lower() if path.suffix else path.name.lower()

# Include recognised expected collections (including extensionless files),
# plus any additional CSVs so unexpected tables are visible too.
collection_paths = [
    p for p in all_data_files
    if normalised_stem(p) in EXPECTED_COLLECTION_STEMS or p.suffix.lower() == ".csv"
]

inventory_rows = []
for path in all_data_files:
    inventory_rows.append({
        "file": path.name,
        "extension": path.suffix or "(none)",
        "size_bytes": path.stat().st_size,
        "recognised_collection": normalised_stem(path) in EXPECTED_COLLECTION_STEMS,
    })

file_inventory = pd.DataFrame(inventory_rows)
display(file_inventory)

found_stems = {normalised_stem(p) for p in collection_paths}
missing_expected = [stem for stem in EXPECTED_COLLECTION_STEMS if stem not in found_stems]
unexpected_collections = [
    p.name for p in collection_paths if normalised_stem(p) not in EXPECTED_COLLECTION_STEMS
]

print("Expected collections missing:", missing_expected if missing_expected else "None")
print("Additional CSV collections:", unexpected_collections if unexpected_collections else "None")

if not collection_paths:
    raise FileNotFoundError(
        f"No expected data collections or CSV files were found in {DATA_DIR}. "
        "Check that the dataset's data/ folder was extracted correctly."
    )

file_inventory.to_csv(TABLES_DIR / "raw_file_inventory.csv", index=False)
print("Inventory saved to:", TABLES_DIR / "raw_file_inventory.csv")

,file,extension,size_bytes,recognised_collection
0,ecg.csv,.csv,4801152,True
1,eda.csv,.csv,1823966,True
2,fitbit_heart_rate_per_minute.csv,.csv,166002,True
3,fitbit_heart_rate_per_second.csv,.csv,4242562,True
4,fitbit_heart_rate_summary.csv,.csv,23799,True
5,participants.csv,.csv,6437,True
6,pss_14.csv,.csv,17527,True
7,stai.csv,.csv,47345,True


Expected collections missing: None
Additional CSV collections: None
Inventory saved to: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\tables\raw_file_inventory.csv


In [10]:
# CELL 3 — load recognised data collections
# Identifiers stay as strings so pseudonymous IDs are not converted to numbers.
# This cell reads data into memory only; it does not rewrite source files.

def read_collection(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(
            path,
            dtype={"username": "string", "participant_id": "string", "subject_id": "string"},
            low_memory=False,
            encoding="utf-8-sig",
        )
    except Exception as exc:
        raise RuntimeError(
            f"Could not parse {path.name} as a CSV table.\n"
            "The file has not been modified. Inspect its format before changing the loader."
        ) from exc


data = {}
source_paths = {}
for path in collection_paths:
    key = normalised_stem(path)
    if key in data:
        raise RuntimeError(
            f"More than one file maps to collection '{key}': "
            f"{source_paths[key].name} and {path.name}. Resolve the duplicate file names first."
        )
    data[key] = read_collection(path)
    source_paths[key] = path

load_rows = []
for name, df in data.items():
    id_candidates = [c for c in ("username", "participant_id", "subject_id", "participant") if c in df.columns]
    id_col = id_candidates[0] if id_candidates else None
    load_rows.append({
        "collection": name,
        "source_file": source_paths[name].name,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "participant_id_column": id_col,
        "unique_participants": int(df[id_col].nunique(dropna=True)) if id_col else np.nan,
        "duplicate_rows": int(df.duplicated().sum()),
    })

collection_summary = pd.DataFrame(load_rows).sort_values("collection").reset_index(drop=True)
display(collection_summary)
collection_summary.to_csv(TABLES_DIR / "collection_summary.csv", index=False)
print(f"Loaded {len(data)} collection(s).")

,collection,source_file,rows,columns,participant_id_column,unique_participants,duplicate_rows
0,ecg,ecg.csv,132,13,username,66,0
1,eda,eda.csv,20094,6,username,64,1542
2,fitbit_heart_rate_per_minute,fitbit_heart_rate_per_minute.csv,2131,3,username,66,0
3,fitbit_heart_rate_per_second,fitbit_heart_rate_per_second.csv,54462,3,username,66,0
4,fitbit_heart_rate_summary,fitbit_heart_rate_summary.csv,264,6,username,66,0
5,participants,participants.csv,66,5,username,66,0
6,pss_14,pss_14.csv,66,17,username,66,0
7,stai,stai.csv,132,24,username,66,0


Loaded 8 collection(s).


In [12]:
# CELL 4 — per-column schema and missingness

schema_rows = []
for name, df in sorted(data.items()):
    for column in df.columns:
        series = df[column]
        schema_rows.append({
            "collection": name,
            "column": column,
            "dtype": str(series.dtype),
            "non_null": int(series.notna().sum()),
            "missing": int(series.isna().sum()),
            "missing_pct": round(float(series.isna().mean() * 100), 3),
            "unique_non_null": int(series.nunique(dropna=True)),
        })

column_schema = pd.DataFrame(schema_rows)
display(column_schema)
column_schema.to_csv(TABLES_DIR / "column_schema_and_missingness.csv", index=False)
print("Saved column schema and missingness table.")

,collection,column,dtype,non_null,missing,missing_pct,unique_non_null
0,ecg,_id,str,132,0,0.0,132
1,ecg,reading_id,str,132,0,0.0,130
2,ecg,reading_time,str,132,0,0.0,132
3,ecg,wire_id,str,132,0,0.0,2
4,ecg,result_classification,str,132,0,0.0,2
...,...,...,...,...,...,...,...
72,stai,"""I feel strained""",str,132,0,0.0,4
73,stai,I feel nervous,str,132,0,0.0,4
74,stai,I feel satisfied,str,132,0,0.0,4
75,stai,I feel secure,str,132,0,0.0,3


Saved column schema and missingness table.


In [14]:
# CELL 5 — participant inventory
# Participant IDs are used only for grouping/auditing, never as model features.

ID_CANDIDATES = ["username", "participant_id", "subject_id", "participant"]
participant_sets = {}
participant_id_columns = {}

for name, df in data.items():
    id_col = next((col for col in ID_CANDIDATES if col in df.columns), None)
    if id_col is not None:
        participant_id_columns[name] = id_col
        participant_sets[name] = set(df[id_col].dropna().astype("string").astype(str).unique())

all_participants = sorted(set().union(*participant_sets.values())) if participant_sets else []
participant_rows = []
for participant in all_participants:
    row = {"participant_id": participant}
    for name, df in sorted(data.items()):
        id_col = participant_id_columns.get(name)
        if id_col is None:
            continue
        mask = df[id_col].astype("string").astype("string") == participant
        participant_rows_count = int(mask.sum())
        row[f"{name}_rows"] = participant_rows_count
        if "phase" in df.columns:
            phases = sorted(
                df.loc[mask, "phase"].dropna().astype("string").str.strip().str.upper().unique().tolist()
            )
            row[f"{name}_phases"] = ";".join(phases)
    participant_rows.append(row)

participant_inventory = pd.DataFrame(participant_rows)
display(participant_inventory.head(20))
print("Distinct participant IDs across collections:", len(all_participants))
print("Collections with a participant ID column:", sorted(participant_sets))
participant_inventory.to_csv(TABLES_DIR / "participant_inventory.csv", index=False)

coverage_rows = []
for name, ids in sorted(participant_sets.items()):
    coverage_rows.append({
        "collection": name,
        "rows": int(len(data[name])),
        "unique_participants": len(ids),
    })
participant_coverage = pd.DataFrame(coverage_rows)
display(participant_coverage)
participant_coverage.to_csv(TABLES_DIR / "participant_coverage.csv", index=False)

,participant_id,ecg_rows,ecg_phases,eda_rows,eda_phases,fitbit_heart_rate_per_minute_rows,fitbit_heart_rate_per_second_rows,fitbit_heart_rate_summary_rows,participants_rows,pss_14_rows,stai_rows,stai_phases
0,02b865b4b8a0f6319640b266de23bd0cfdffc345e75ec64c5772ca659541e3f7,2,RELAX;STRESS,163,RELAX,10,243,4,1,1,2,RELAX;STRESS
1,036e6da345803c97c92d64a489ecdfefd93adb7cd9474322e3fdc455164372da,2,RELAX;STRESS,326,RELAX;STRESS,32,810,4,1,1,2,RELAX;STRESS
2,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e194be517ee7d28b68c4,2,RELAX;STRESS,325,RELAX;STRESS,36,891,4,1,1,2,RELAX;STRESS
3,09223ce6fe3b28d121bf18df0e4e8f09622209678abf7b706bc780646dbf2d53,2,RELAX;STRESS,322,RELAX;STRESS,36,717,4,1,1,2,RELAX;STRESS
4,0e1d276d538d521abc42245a3920f6282c3279e0337caf7d3435fa0c424db705,2,RELAX;STRESS,321,RELAX;STRESS,31,745,4,1,1,2,RELAX;STRESS
5,0f53170a435a716c32ea8392d549ba057b4e29c50bb12f01eb9f99f643134f7c,2,RELAX;STRESS,321,RELAX;STRESS,30,818,4,1,1,2,RELAX;STRESS
6,14bdb864e416460c7a92da08570929fe3d80a8688c803830205ca95f71620756,2,RELAX;STRESS,324,RELAX;STRESS,29,752,4,1,1,2,RELAX;STRESS
7,160146850768697cf9984655ffa2f956f10a1e41b1c7f2d04e2f0c54db409811,2,RELAX;STRESS,326,RELAX;STRESS,31,746,4,1,1,2,RELAX;STRESS
8,17fc54dbc1172a9c00cc8a931412bc0b65dc3b51507cbbf5c87db086c8f925c3,2,RELAX;STRESS,322,RELAX;STRESS,40,1064,4,1,1,2,RELAX;STRESS
9,23d08053659320558c3ceb47cd928d8ef2d07a4cc0eeb29a9bca991dc844c886,2,RELAX;STRESS,326,RELAX;STRESS,33,896,4,1,1,2,RELAX;STRESS


Distinct participant IDs across collections: 66
Collections with a participant ID column: ['ecg', 'eda', 'fitbit_heart_rate_per_minute', 'fitbit_heart_rate_per_second', 'fitbit_heart_rate_summary', 'participants', 'pss_14', 'stai']


,collection,rows,unique_participants
0,ecg,132,66
1,eda,20094,64
2,fitbit_heart_rate_per_minute,2131,66
3,fitbit_heart_rate_per_second,54462,66
4,fitbit_heart_rate_summary,264,66
5,participants,66,66
6,pss_14,66,66
7,stai,132,66


In [15]:
# CELL 6 — phase label counts and participant × phase coverage

phase_rows = []
for name, df in sorted(data.items()):
    if "phase" not in df.columns:
        continue
    labels = df["phase"].astype("string").str.strip().str.upper()
    counts = labels.value_counts(dropna=False)
    for label, count in counts.items():
        phase_rows.append({
            "collection": name,
            "phase": "<MISSING>" if pd.isna(label) else str(label),
            "records": int(count),
        })

phase_distribution = pd.DataFrame(phase_rows)
display(phase_distribution)
phase_distribution.to_csv(TABLES_DIR / "phase_label_distribution.csv", index=False)

for name in ("ecg", "eda", "stai", "fitbit_heart_rate_per_second", "fitbit_heart_rate_per_minute"):
    if name not in data:
        continue
    df = data[name]
    id_col = next((c for c in ID_CANDIDATES if c in df.columns), None)
    if id_col and "phase" in df.columns:
        print(f"\n{name}: participant × phase record counts")
        cross_tab = pd.crosstab(
            df[id_col].astype("string"),
            df["phase"].astype("string").str.strip().str.upper(),
            margins=True,
        )
        display(cross_tab)
        cross_tab.to_csv(TABLES_DIR / f"{name}_participant_phase_crosstab.csv")

,collection,phase,records
0,ecg,RELAX,66
1,ecg,STRESS,66
2,eda,RELAX,10367
3,eda,STRESS,9727
4,stai,RELAX,66
5,stai,STRESS,66



ecg: participant × phase record counts


phase,RELAX,STRESS,All
username,,,
02b865b4b8a0f6319640b266de23bd0cfdffc345e75ec64c5772ca659541e3f7,1,1,2
036e6da345803c97c92d64a489ecdfefd93adb7cd9474322e3fdc455164372da,1,1,2
06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e194be517ee7d28b68c4,1,1,2
09223ce6fe3b28d121bf18df0e4e8f09622209678abf7b706bc780646dbf2d53,1,1,2
0e1d276d538d521abc42245a3920f6282c3279e0337caf7d3435fa0c424db705,1,1,2
...,...,...,...
ef6f580a11d00b211e650fa950261d7cde308adbe65f2fd70773665de698882c,1,1,2
f451d4649a36bc0882a3ad1b60aaa612c7f955d53d57f95e8109d8a1c75acdcf,1,1,2
f9273aee01ee432d04ed2f4f65d247c947e9111a06c563f387408d50260db59e,1,1,2



eda: participant × phase record counts


phase,RELAX,STRESS,All
username,,,
02b865b4b8a0f6319640b266de23bd0cfdffc345e75ec64c5772ca659541e3f7,163,0,163
036e6da345803c97c92d64a489ecdfefd93adb7cd9474322e3fdc455164372da,163,163,326
06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e194be517ee7d28b68c4,163,162,325
09223ce6fe3b28d121bf18df0e4e8f09622209678abf7b706bc780646dbf2d53,160,162,322
0e1d276d538d521abc42245a3920f6282c3279e0337caf7d3435fa0c424db705,158,163,321
...,...,...,...
ef6f580a11d00b211e650fa950261d7cde308adbe65f2fd70773665de698882c,163,163,326
f451d4649a36bc0882a3ad1b60aaa612c7f955d53d57f95e8109d8a1c75acdcf,163,162,325
f9273aee01ee432d04ed2f4f65d247c947e9111a06c563f387408d50260db59e,161,159,320



stai: participant × phase record counts


phase,RELAX,STRESS,All
username,,,
02b865b4b8a0f6319640b266de23bd0cfdffc345e75ec64c5772ca659541e3f7,1,1,2
036e6da345803c97c92d64a489ecdfefd93adb7cd9474322e3fdc455164372da,1,1,2
06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e194be517ee7d28b68c4,1,1,2
09223ce6fe3b28d121bf18df0e4e8f09622209678abf7b706bc780646dbf2d53,1,1,2
0e1d276d538d521abc42245a3920f6282c3279e0337caf7d3435fa0c424db705,1,1,2
...,...,...,...
ef6f580a11d00b211e650fa950261d7cde308adbe65f2fd70773665de698882c,1,1,2
f451d4649a36bc0882a3ad1b60aaa612c7f955d53d57f95e8109d8a1c75acdcf,1,1,2
f9273aee01ee432d04ed2f4f65d247c947e9111a06c563f387408d50260db59e,1,1,2


In [16]:
# CELL 7 — EDA markers, validity flags, and ECG waveform field structure
# These are diagnostics only. We do not resample, interpolate, or parse waveforms here.

if "eda" in data:
    eda = data["eda"]
    print("EDA columns:", eda.columns.tolist())
    if "time_sec" in eda.columns:
        time_values = pd.to_numeric(eda["time_sec"], errors="coerce")
        print({
            "rows": int(len(time_values)),
            "numeric_time_values": int(time_values.notna().sum()),
            "unique_time_markers": int(time_values.nunique(dropna=True)),
            "repeated_time_markers": int(time_values.duplicated(keep=False).sum()),
            "missing_or_non_numeric": int(time_values.isna().sum()),
            "minimum": float(time_values.min()) if time_values.notna().any() else np.nan,
            "maximum": float(time_values.max()) if time_values.notna().any() else np.nan,
        })
    if "valid_data" in eda.columns:
        print("EDA valid_data values:")
        display(eda["valid_data"].value_counts(dropna=False).rename_axis("valid_data").reset_index(name="records"))

if "ecg" in data:
    ecg = data["ecg"]
    print("\nECG columns:", ecg.columns.tolist())
    if "waveform_samples" in ecg.columns:
        waveform = ecg["waveform_samples"].dropna().astype(str)
        waveform_lengths = waveform.str.len()
        print("ECG waveform_samples field diagnostics:")
        display(pd.DataFrame({
            "non_null_waveform_records": [int(len(waveform))],
            "min_characters": [int(waveform_lengths.min()) if len(waveform_lengths) else np.nan],
            "median_characters": [float(waveform_lengths.median()) if len(waveform_lengths) else np.nan],
            "max_characters": [int(waveform_lengths.max()) if len(waveform_lengths) else np.nan],
        }))
        preview_cols = [c for c in ("username", "phase", "waveform_samples") if c in ecg.columns]
        if preview_cols:
            preview = ecg[preview_cols].head(3).copy()
            if "waveform_samples" in preview.columns:
                preview["waveform_samples"] = preview["waveform_samples"].astype("string").str.slice(0, 180)
            display(preview)
        print("Do not infer ECG units or parse delimiters from this preview. Verify the dataset documentation first.")

for name, df in sorted(data.items()):
    if "valid_data" in df.columns:
        print(f"\n{name}: valid_data frequency")
        display(df["valid_data"].value_counts(dropna=False).rename_axis("valid_data").reset_index(name="records"))

EDA columns: ['username', 'phase', 'time_sec', 'valid_data', 'activation', 'scl_avg']
{'rows': 20094, 'numeric_time_values': 20094, 'unique_time_markers': 128, 'repeated_time_markers': 20094, 'missing_or_non_numeric': 0, 'minimum': -3494.0, 'maximum': 163.0}
EDA valid_data values:


,valid_data,records
0,True,19708
1,False,386



ECG columns: ['_id', 'reading_id', 'reading_time', 'wire_id', 'result_classification', 'heart_rate', 'heart_rate_alert', 'firmware_version', 'device_app_version', 'hardware_version', 'username', 'phase', 'waveform_samples']
ECG waveform_samples field diagnostics:


,non_null_waveform_records,min_characters,median_characters,max_characters
0,132,33950,36005.0,39625


,username,phase,waveform_samples
0,6683f97240e6fd3620594043bee323169ed91d789c7e996e78dd3b461e90c1c4,relax,3897 4296 4802 5377 5954 6636 7630 8931 10175 10888 10984 10903 11141 11805 12696 13600 14263 14485 14435 14478 1485...
1,6683f97240e6fd3620594043bee323169ed91d789c7e996e78dd3b461e90c1c4,stress,0 353 485 331 10 -316 -599 -886 -1181 -1404 -1496 -1514 -1585 -1793 -2070 -2270 -2372 -2508 -2802 -3240 -3681 -3990 ...
2,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e194be517ee7d28b68c4,relax,-801 -1104 -1342 -1508 -1660 -1888 -2219 -2545 -2757 -2894 -3075 -3343 -3639 -3874 -3995 -4020 -4018 -4049 -4130 -42...


Do not infer ECG units or parse delimiters from this preview. Verify the dataset documentation first.

eda: valid_data frequency


,valid_data,records
0,True,19708
1,False,386


In [17]:
# CELL 8 — descriptive numeric summaries (not clinical range validation)

numeric_summary_frames = []
duplicate_summary = []

for name, df in sorted(data.items()):
    duplicate_summary.append({
        "collection": name,
        "rows": int(len(df)),
        "duplicate_rows": int(df.duplicated().sum()),
    })
    numeric_df = df.select_dtypes(include=["number"])
    if not numeric_df.empty:
        summary = numeric_df.describe().T.reset_index().rename(columns={"index": "column"})
        summary.insert(0, "collection", name)
        numeric_summary_frames.append(summary)

duplicate_summary_df = pd.DataFrame(duplicate_summary)
display(duplicate_summary_df)
duplicate_summary_df.to_csv(TABLES_DIR / "duplicate_row_summary.csv", index=False)

if numeric_summary_frames:
    numeric_summary = pd.concat(numeric_summary_frames, ignore_index=True)
    display(numeric_summary)
    numeric_summary.to_csv(TABLES_DIR / "numeric_summary.csv", index=False)
else:
    numeric_summary = pd.DataFrame()
    print("No numeric columns were detected in the loaded collections.")

print("Reminder: descriptive ranges are not medical thresholds and should not be used to label records as healthy/unhealthy.")

,collection,rows,duplicate_rows
0,ecg,132,0
1,eda,20094,1542
2,fitbit_heart_rate_per_minute,2131,0
3,fitbit_heart_rate_per_second,54462,0
4,fitbit_heart_rate_summary,264,0
5,participants,66,0
6,pss_14,66,0
7,stai,132,0


,collection,column,count,mean,std,min,25%,50%,75%,max
0,ecg,heart_rate,132.0,67.628788,10.307046,41.00,60.00,66.50,74.000000,94.0000
1,ecg,device_app_version,132.0,2.160000,0.000000,2.16,2.16,2.16,2.160000,2.1600
2,eda,time_sec,20094.0,49.516522,336.829681,-3494.00,40.00,81.00,121.000000,163.0000
3,eda,activation,20094.0,0.141336,0.348376,0.00,0.00,0.00,0.000000,1.0000
4,eda,scl_avg,20094.0,38.421071,8.751348,2.21,32.73,39.80,44.470000,56.4600
5,fitbit_heart_rate_per_minute,value,2131.0,72.777100,11.566447,40.00,65.00,73.00,80.000000,139.0000
6,fitbit_heart_rate_per_second,value,54462.0,72.904355,12.216810,39.00,65.00,72.00,80.000000,152.0000
7,fitbit_heart_rate_summary,min,264.0,112.931818,52.466784,30.00,82.50,118.00,143.000000,176.0000
8,fitbit_heart_rate_summary,max,264.0,160.431818,40.481737,100.00,119.50,144.00,187.000000,220.0000
9,fitbit_heart_rate_summary,minutes,264.0,360.000000,624.591279,0.00,0.00,0.00,361.500000,1440.0000


Reminder: descriptive ranges are not medical thresholds and should not be used to label records as healthy/unhealthy.


In [18]:
# CELL 9 — documentation file presence and short previews
# These files are read as text only. The notebook does not modify them.

DOCUMENTATION_FILES = [
    "README.md",
    "LICENSE.txt",
    "PHYSIONET_REPORT.md",
    "data_description.csv",
    "SHA256SUMS.txt",
]

documentation_status = []
for filename in DOCUMENTATION_FILES:
    path = DATASET_ROOT / filename
    documentation_status.append({
        "file": filename,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else np.nan,
    })

documentation_inventory = pd.DataFrame(documentation_status)
display(documentation_inventory)

for filename in ("README.md", "PHYSIONET_REPORT.md", "LICENSE.txt"):
    path = DATASET_ROOT / filename
    if path.is_file():
        print(f"\n--- {filename} (first 35 lines) ---")
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("\n".join(lines[:35]))

# data_description.csv is intentionally previewed, not assumed to follow the same schema as signals.
description_path = DATASET_ROOT / "data_description.csv"
if description_path.is_file():
    print("\ndata_description.csv preview:")
    try:
        description_preview = pd.read_csv(description_path, low_memory=False)
        display(description_preview.head(30))
    except Exception as exc:
        print("Could not read data_description.csv as a table:", repr(exc))

,file,exists,size_bytes
0,README.md,True,3186
1,LICENSE.txt,True,14842
2,PHYSIONET_REPORT.md,True,637
3,data_description.csv,True,567
4,SHA256SUMS.txt,True,1032



--- README.md (first 35 lines) ---
# QoL_Stress Dataset

## Authors
- Amaia Calvo¹*, Ander Cejudo¹², Cristina Martin¹²³  
¹ Fundación Vicomtech, Basque Research and Technology Alliance (BRTA), Donostia-San Sebastián, Spain  
² Faculty of Engineering, University of Deusto, Bilbao, Spain  
³ BioGipuzkoa Health Research Institute, eHealth Group, Donostia-San Sebastián, Spain  

## Overview
The QoL_Stress dataset contains physiological signals, heart rate data from Fitbit devices, and stress/anxiety questionnaire responses from 66 participants. It was collected to study stress and quality-of-life patterns under controlled relaxation and stress conditions. The dataset can be used for stress detection, physiological signal analysis, and machine learning applications.

## Background
Participants underwent periods of relaxation and induced stress while physiological signals were recorded and questionnaires administered. The study focuses on healthy adults and aims to provide data for research

,data_type,description,files
0,time,Timestamp of sample collection,ecg.csv; eda.csv; fitbit_heart_rate_per_minute.csv; fitbit_heart_rate_per_second.csv
1,ecg,Electrocardiogram waveform,ecg.csv
2,eda,Electrodermal activity,eda.csv
3,heart_rate,Heart rate measured per second and per minute,fitbit_heart_rate_per_second.csv; fitbit_heart_rate_per_minute.csv
4,heart_rate_zones,Daily summary of heart rate zones,fitbit_heart_rate_summary.csv
5,questionnaires,Perceived stress and anxiety scores,pss_14.csv; stai.csv
6,demographics,Participant demographic data,participants.csv


In [19]:
# CELL 10 — checksum verification
# Hashing can take a little time for large files. Only reads files; writes a results table.


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


checksum_path = DATASET_ROOT / "SHA256SUMS.txt"
checksum_rows = []

if not checksum_path.is_file():
    print("SHA256SUMS.txt was not found. Checksum verification skipped.")
else:
    checksum_pattern = re.compile(r"^([A-Fa-f0-9]{64})\s+[* ]?(.*)$")
    for raw_line in checksum_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        match = checksum_pattern.match(line)
        if not match:
            checksum_rows.append({"entry": line, "status": "UNPARSED_LINE"})
            continue

        expected_hash, relative_name = match.groups()
        candidate = (DATASET_ROOT / relative_name).resolve()
        try:
            candidate.relative_to(DATASET_ROOT.resolve())
        except ValueError:
            checksum_rows.append({"file": relative_name, "status": "PATH_OUTSIDE_DATASET"})
            continue

        if not candidate.is_file():
            checksum_rows.append({"file": relative_name, "expected_sha256": expected_hash, "status": "MISSING_FILE"})
            continue

        actual_hash = sha256_file(candidate)
        checksum_rows.append({
            "file": relative_name,
            "expected_sha256": expected_hash,
            "actual_sha256": actual_hash,
            "status": "OK" if actual_hash.lower() == expected_hash.lower() else "HASH_MISMATCH",
        })

checksum_results = pd.DataFrame(checksum_rows)
display(checksum_results)
checksum_results.to_csv(TABLES_DIR / "checksum_verification.csv", index=False)
if not checksum_rows:
    print("No checksum entries were processed.")
else:
    print("Checksum status counts:")
    display(checksum_results["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="entries"))

,file,expected_sha256,actual_sha256,status
0,LICENSE.txt,9a78e7f22742dde9f66ae235ec793ba2212019dc4d0ced75c4da09ced0b35fb2,9a78e7f22742dde9f66ae235ec793ba2212019dc4d0ced75c4da09ced0b35fb2,OK
1,PHYSIONET_REPORT.md,0ac23769b95b465629b4d9e064555585b3e75adf944a17eebbb0db457c51e189,0ac23769b95b465629b4d9e064555585b3e75adf944a17eebbb0db457c51e189,OK
2,README.md,db7131157f633ce49dcca21939f4b38a71ebfc2d86b571980623f931148de39a,db7131157f633ce49dcca21939f4b38a71ebfc2d86b571980623f931148de39a,OK
3,data/ecg.csv,a8c668be74859a816d299b450b5094f7c47d9f0b97d3691aa610ed37b9e8fcf5,a8c668be74859a816d299b450b5094f7c47d9f0b97d3691aa610ed37b9e8fcf5,OK
4,data/eda.csv,cef959b2631cd13d43d059f2c462d8be4727fbeb4cab2a10723667e287fee134,cef959b2631cd13d43d059f2c462d8be4727fbeb4cab2a10723667e287fee134,OK
5,data/fitbit_heart_rate_per_minute.csv,31577912bd873edd40825ed8811323052f0376d92a9bdce1d00603ccce2c5b34,31577912bd873edd40825ed8811323052f0376d92a9bdce1d00603ccce2c5b34,OK
6,data/fitbit_heart_rate_per_second.csv,b8b69ebbc2a2e2b13aa366d75e4ceeb4d3e698a652644a6861e820ec95ea7ccd,b8b69ebbc2a2e2b13aa366d75e4ceeb4d3e698a652644a6861e820ec95ea7ccd,OK
7,data/fitbit_heart_rate_summary.csv,642fee88e04de1d96555eaf4749dd59c679495e7036b756aa9c97fdc583a9ff1,642fee88e04de1d96555eaf4749dd59c679495e7036b756aa9c97fdc583a9ff1,OK
8,data/participants.csv,fb11443552dcbc1e5a7ba63c2a33ab07a1f3a5a7a661aecde6db084a01abff27,fb11443552dcbc1e5a7ba63c2a33ab07a1f3a5a7a661aecde6db084a01abff27,OK
9,data/pss_14.csv,1b66de218a7d948959daa6356e3417ce1124adfb2c736cce950c5ed0bf40034e,1b66de218a7d948959daa6356e3417ce1124adfb2c736cce950c5ed0bf40034e,OK


Checksum status counts:


,status,entries
0,OK,12


In [20]:
# CELL 11 — write summary reports
# All output destinations are absolute paths rooted at PROJECT_ROOT.

expected_missing_text = ", ".join(missing_expected) if missing_expected else "None"
unexpected_text = ", ".join(unexpected_collections) if unexpected_collections else "None"
checksum_mismatch_count = (
    int((checksum_results["status"] == "HASH_MISMATCH").sum())
    if not checksum_results.empty and "status" in checksum_results.columns else 0
)

report_lines = [
    "# QoL_Stress dataset audit",
    "",
    "This report was generated by `notebooks/01_dataset_audit_corrected.ipynb`.",
    "It reports structural properties observed in the local files; it does not train a model or infer clinical status.",
    "",
    f"- Project root: `{PROJECT_ROOT}`",
    f"- Dataset root: `{DATASET_ROOT}`",
    f"- Data directory: `{DATA_DIR}`",
    f"- Collections loaded: {len(data)}",
    f"- Expected collections missing: {expected_missing_text}",
    f"- Additional CSV collections: {unexpected_text}",
    f"- Distinct participant IDs across collections: {len(all_participants)}",
    f"- Checksum mismatches: {checksum_mismatch_count}",
    "",
    "## Collection summary",
    "",
    "| Collection | Source file | Rows | Columns | Participants | Duplicate rows |",
    "|---|---|---:|---:|---:|---:|",
]
for row in collection_summary.to_dict(orient="records"):
    participants = "n/a" if pd.isna(row["unique_participants"]) else int(row["unique_participants"])
    report_lines.append(
        f"| {row['collection']} | {row['source_file']} | {row['rows']} | {row['columns']} | {participants} | {row['duplicate_rows']} |"
    )

report_lines += [
    "",
    "## Files written by this audit",
    "",
    "- `reports/data_audit.md`",
    "- `reports/data_quality_report.md`",
    "- `reports/file_inventory.json`",
    "- `reports/tables/raw_file_inventory.csv`",
    "- `reports/tables/collection_summary.csv`",
    "- `reports/tables/column_schema_and_missingness.csv`",
    "- `reports/tables/participant_inventory.csv`",
    "- `reports/tables/participant_coverage.csv`",
    "- `reports/tables/phase_label_distribution.csv`",
    "- `reports/tables/duplicate_row_summary.csv`",
    "- `reports/tables/numeric_summary.csv` (when numeric columns exist)",
    "- `reports/tables/checksum_verification.csv`",
    "",
    "## Required review before modelling",
    "",
    "1. Confirm the meaning of one row in each collection from the official documentation.",
    "2. Confirm the experimental phase label semantics and whether each participant has both phases for the chosen modality.",
    "3. Investigate missing, invalid, and duplicate records; do not automatically delete them.",
    "4. Confirm the units, sampling frequency, time origin, and waveform encoding before extracting signal features.",
    "5. Determine whether modality records can be paired by participant, phase, and time; do not assume synchronisation.",
    "6. Use participant-grouped validation (for example, Leave-One-Subject-Out or GroupKFold) for future modelling.",
    "7. Keep participant IDs out of the predictive feature matrix.",
    "",
    "## Scientific limitations",
    "",
    "- RELAX/STRESS are controlled experimental conditions, not clinical diagnoses or measures of chronic stress.",
    "- Results from this dataset cannot by themselves establish astronaut-specific stress detection or clinical validity.",
    "- Descriptive statistics in this report are not clinical thresholds.",
    "- Missing or invalid recordings are not negative labels.",
]

(REPORTS_DIR / "data_audit.md").write_text("\n".join(report_lines), encoding="utf-8")

quality_lines = [
    "# Data quality report",
    "",
    f"Dataset root: `{DATASET_ROOT}`",
    "",
    "## Missing expected collections",
    "",
    expected_missing_text,
    "",
    "## Participant coverage",
    "",
    f"Distinct participant IDs across all collections with an ID field: {len(all_participants)}",
    "",
    "## Collection-specific observations",
    "",
]
for name, df in sorted(data.items()):
    quality_lines.append(f"### {name}")
    quality_lines.append("")
    quality_lines.append(f"- Rows: {len(df)}")
    quality_lines.append(f"- Columns: {len(df.columns)}")
    quality_lines.append(f"- Duplicate rows: {int(df.duplicated().sum())}")
    id_col = participant_id_columns.get(name)
    if id_col:
        quality_lines.append(f"- Participant ID field: `{id_col}`")
        quality_lines.append(f"- Unique participants: {int(df[id_col].nunique(dropna=True))}")
    if "phase" in df.columns:
        phase_counts = df["phase"].astype("string").str.strip().str.upper().value_counts(dropna=False)
        quality_lines.append("- Phase counts:")
        for label, count in phase_counts.items():
            label_text = "<MISSING>" if pd.isna(label) else str(label)
            quality_lines.append(f"  - {label_text}: {int(count)}")
    if "valid_data" in df.columns:
        valid_counts = df["valid_data"].value_counts(dropna=False)
        quality_lines.append("- `valid_data` values:")
        for label, count in valid_counts.items():
            quality_lines.append(f"  - {label}: {int(count)}")
    if "time_sec" in df.columns:
        time_values = pd.to_numeric(df["time_sec"], errors="coerce")
        quality_lines.append(
            f"- `time_sec`: {int(time_values.notna().sum())} numeric values; "
            f"{int(time_values.nunique(dropna=True))} unique markers; "
            f"{int(time_values.duplicated(keep=False).sum())} rows participate in repeated markers."
        )
    quality_lines.append("")

quality_lines += [
    "## Interpretation cautions",
    "",
    "- Do not assume `time_sec` is a uniform one-second sampling clock.",
    "- Do not silently interpolate or resample gaps.",
    "- Do not assume that every modality has a complete matching record for every participant and phase.",
    "- Do not interpret experimental phase labels as clinical outcomes.",
]
(REPORTS_DIR / "data_quality_report.md").write_text("\n".join(quality_lines), encoding="utf-8")

file_inventory.to_json(REPORTS_DIR / "file_inventory.json", orient="records", indent=2)

print("Audit reports generated successfully.")
print("Data audit report :", REPORTS_DIR / "data_audit.md")
print("Quality report    :", REPORTS_DIR / "data_quality_report.md")
print("Tables            :", TABLES_DIR)
print("Raw source files were not modified.")

Audit reports generated successfully.
Data audit report : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\data_audit.md
Quality report    : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\data_quality_report.md
Tables            : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\tables
Raw source files were not modified.


## Audit interpretation — complete after reviewing outputs

Do not begin model training until you have answered: (1) which modalities have both RELAX and STRESS data per participant; (2) what rows actually represent in each table; (3) how the phase label is assigned; (4) what missingness means; (5) which time series can legitimately be aligned; and (6) what is the independent evaluation unit.

For this dataset, subject-independent evaluation is essential: keep all records/windows from one participant together in a held-out fold. Do not use `username` as a model feature. Do not claim chronic astronaut stress detection from this controlled protocol.